# Project 06 — BROKEN notebook (debugging exercise)

This notebook forces a **Poisson** model on overdispersed data and then fails to notice. Run it, read the posterior-predictive check, find each bug, and fix it. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']
print('var/mean =', y.var()/y.mean(), '(>>1 => overdispersed!)')

### Model — BUG 1: a Poisson likelihood on data we just saw are overdispersed.

In [ ]:
# BUG 1: Poisson forces Var(y)=mu. The data have var/mean ~ 14, so this is
#        the wrong family. The fit will CONVERGE (don't be fooled) but its
#        predictions will be far too tight.
with pm.Model() as model:
    beta0 = pm.Normal('beta0', 0.0, 2.0)
    beta1 = pm.Normal('beta1', 0.0, 1.0)
    mu = pm.math.exp(beta0 + beta1 * x)
    pm.Poisson('y', mu=mu, observed=y)
    idata = pm.sample(draws=800, tune=800, chains=2, random_seed=RNG,
                      progressbar=False, idata_kwargs={'log_likelihood': True})
    idata.extend(pm.sample_posterior_predictive(idata, random_seed=RNG,
                                                progressbar=False))

In [ ]:
# Converges fine -> easy to declare victory here. That is the trap.
print(az.summary(idata, var_names=['beta0', 'beta1']))

### Criticism — BUG 2: 'checking' only R-hat, never a PPC. The misspecification is invisible to convergence diagnostics.

In [ ]:
# BUG 2: stopping at R-hat. Convergence != adequacy. A correct workflow runs
#        a posterior-predictive check; here is the check that WOULD reveal the
#        problem (predicted variance << observed variance). Uncomment-style fix:
ppy = idata.posterior_predictive['y'].values.reshape(-1, len(y))
print('observed var =', round(float(y.var()), 1),
      'predicted var ~', round(float(ppy.var(axis=1).mean()), 1))
# The two numbers are wildly different -> the Poisson is under-dispersed.
# BUG 3 (implicit): there is no second model to compare against. Fit a
# NegativeBinomial and run az.compare to see it win.